## 「前置硬清洗剥离」和「分层抽样保证违约比例」：
删除重复样本、逻辑错误样本、完全无效样本；

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from sklearn.model_selection import train_test_split  # 分层抽样依赖库

# 设置中文显示，避免画图乱码
plt.rcParams["font.family"] = "Microsoft YaHei"
plt.rcParams["axes.unicode_minus"] = False

## 创建输出文件夹和加载数据字典

In [2]:
# 创建数据存储路径（processed存清洗/划分后的数据，output-img存图表）
os.makedirs("../data/raw", exist_ok=True)    # 原始数据路径（确保cs-training.csv在此）
os.makedirs("../data/processed", exist_ok=True)   # 清洗+划分后的数据路径
os.makedirs("../data/output-img", exist_ok=True)  # 可视化图片路径

print("✅ 输出文件夹创建/检查完成！")

# ！！！关键：直接指定违约目标变量（金融数据GCD数据集默认是SeriousDlqin2yrs）
# 含义：2年内是否严重违约，1=违约，0=正常
target_col = "SeriousDlqin2yrs"
print(f"\n🎯 确认分层抽样目标变量：{target_col}（2年内是否违约，1=违约，0=正常）")

✅ 输出文件夹创建/检查完成！

🎯 确认分层抽样目标变量：SeriousDlqin2yrs（2年内是否违约，1=违约，0=正常）


## 加载原始数据集 + 前置硬清洗
针对 逾期天数（30-59，60-89，大于90）
  - 出现割裂的96和98的数值，与其他值差距极大
  - 考虑到98常作为缺失值编码（如98表示未知）

针对 age 小于18
- 数据少，所以可以做删除的处理

针对 月收入 大于100万
- 考虑到只有3个数据，并且都未违约
- 考虑到极端值对DNN训练的影响很大
- 所以直接删去

In [3]:
# 1. 加载原始数据
# 这里不设置 index_col，先完整读入，再显式处理首列 id
df_raw = pd.read_csv("../data/raw/cs-training.csv")

# 1.1 删除首列 id（如果首列是编号列）
first_col = df_raw.columns[0]
first_col_series = df_raw[first_col]

is_integer_like = pd.api.types.is_integer_dtype(first_col_series)
is_unique = first_col_series.nunique(dropna=False) == len(df_raw)
name_like_id = str(first_col).strip().lower() in {"id", "unnamed: 0"}

if (name_like_id and is_unique) or (is_integer_like and is_unique):
    df_raw = df_raw.iloc[:, 1:]
    print(f"🧹 已删除首列ID字段：{first_col}")
else:
    print(f"ℹ️ 首列未识别为ID字段，保留：{first_col}")

print("="*60)
print("✅ 原始数据加载成功！")
print(f"原始数据形状：{df_raw.shape[0]} 行，{df_raw.shape[1]} 列")
print("="*60)

# 2. 前置硬清洗：仅删除「无统计依赖的硬错误」（已移除重复数据处理，新增月收入极端值删除）
df_cleaned = df_raw.copy()
original_count = len(df_cleaned)

# 2.1 删除逾期字段=96/98的异常样本（业务规则：96/98为无效编码）
due_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]
mask_due = False
for col in due_cols:
    mask_due |= (df_cleaned[col] == 96) | (df_cleaned[col] == 98)
due_delete_count = mask_due.sum()
df_cleaned = df_cleaned[~mask_due]
print(f"\n1. 逾期异常值处理：")
print(f"   删除逾期字段=96/98的样本数：{due_delete_count} 条")

# 2.2 删除年龄<18岁的无效样本（业务规则：未成年人无信贷资格）
mask_age = df_cleaned["age"] < 18
age_delete_count = mask_age.sum()
df_cleaned = df_cleaned[~mask_age]
print(f"\n2. 年龄异常值处理：")
print(f"   删除年龄<18岁的样本数：{age_delete_count} 条")

# 2.3 新增：删除月收入>100万的极端异常样本（业务规则：远超正常信贷用户收入范围）
# 先处理可能的缺失值（避免NaN被误判为>100万，NaN后续填充阶段处理）
mask_income = df_cleaned["MonthlyIncome"] > 1000000  # 100万=1000000
income_delete_count = mask_income.sum()
df_cleaned = df_cleaned[~mask_income]
print(f"\n3. 月收入极端值处理：")
print(f"   删除月收入>100万的样本数：{income_delete_count} 条")

# 3. 硬清洗结果总结（同步更新步骤统计）
print("\n" + "="*60)
print("✅ 前置硬清洗完成！")
print(f"原始样本数：{original_count} 条")
print(f"清洗后样本数：{len(df_cleaned)} 条")
# 累计删除数 = 逾期删除数 + 年龄删除数 + 月收入删除数
total_delete_count = due_delete_count + age_delete_count + income_delete_count
print(f"累计删除硬错误样本：{total_delete_count} 条")
print(f"  - 逾期异常：{due_delete_count} 条")
print(f"  - 年龄无效：{age_delete_count} 条")
print(f"  - 月收入>100万：{income_delete_count} 条")
print(f"清洗后违约比例：{df_cleaned[target_col].mean():.2%}（与后续分层抽样基准一致）")
print("="*60)

🧹 已删除首列ID字段：Unnamed: 0
✅ 原始数据加载成功！
原始数据形状：150000 行，11 列

1. 逾期异常值处理：
   删除逾期字段=96/98的样本数：269 条

2. 年龄异常值处理：
   删除年龄<18岁的样本数：1 条

3. 月收入极端值处理：
   删除月收入>100万的样本数：4 条

✅ 前置硬清洗完成！
原始样本数：150000 条
清洗后样本数：149726 条
累计删除硬错误样本：274 条
  - 逾期异常：269 条
  - 年龄无效：1 条
  - 月收入>100万：4 条
清洗后违约比例：6.60%（与后续分层抽样基准一致）


## 分层抽样（保证训练 / 测试集违约比例与原始一致）

In [4]:
# 1. 分层抽样参数设置
test_size = 0.2  # 测试集占比（可调整，如0.3）
random_state = 42  # 固定随机种子，确保结果可复现

# 2. 执行分层抽样（stratify参数强制按目标变量比例划分）
X = df_cleaned.drop(columns=[target_col])  # 特征矩阵
y = df_cleaned[target_col]                 # 目标变量（违约标识）

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size,
    random_state=random_state,
    stratify=y  # 核心：按目标变量分层，保证训练/测试集违约比例一致
)

# 3. 合并特征与目标变量，形成最终训练/测试集
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

# 4. 验证分层抽样效果（检查违约比例是否一致）
print("📊 分层抽样结果验证：")
print(f"原始清洗后数据违约比例：{df_cleaned[target_col].mean():.2%}")
print(f"训练集违约比例：{train_df[target_col].mean():.2%}")
print(f"测试集违约比例：{test_df[target_col].mean():.2%}")
print(f"\n数据量分布：")
print(f"训练集：{train_df.shape[0]} 行（{1-test_size:.0%}）")
print(f"测试集：{test_df.shape[0]} 行（{test_size:.0%}）")
print(f"总样本数：{train_df.shape[0] + test_df.shape[0]} 行（与清洗后一致）")

📊 分层抽样结果验证：
原始清洗后数据违约比例：6.60%
训练集违约比例：6.60%
测试集违约比例：6.60%

数据量分布：
训练集：119780 行（80%）
测试集：29946 行（20%）
总样本数：149726 行（与清洗后一致）


## 保存分层后的训练 / 测试集

In [5]:
# 保存训练集和测试集到processed文件夹（后续预处理仅用训练集拟合参数）
# 不保存DataFrame索引，避免额外的索引列写入CSV
train_df.to_csv("../data/processed/train_set.csv", index=False)
test_df.to_csv("../data/processed/test_set.csv", index=False)

print("=" * 60)
print("✅ 分层抽样数据保存完成！")
print(f"训练集路径：../data/processed/train_set.csv")
print(f"测试集路径：../data/processed/test_set.csv")
print("\n⚠️  后续注意：")
print("1. 缺失值填充、异常值截尾（如DebtRatio）需用【训练集】拟合参数")
print("2. 测试集仅用于最终评估，不参与任何参数拟合")
print("=" * 60)

✅ 分层抽样数据保存完成！
训练集路径：../data/processed/train_set.csv
测试集路径：../data/processed/test_set.csv

⚠️  后续注意：
1. 缺失值填充、异常值截尾（如DebtRatio）需用【训练集】拟合参数
2. 测试集仅用于最终评估，不参与任何参数拟合
